# Profiling 实操：基线 vs 融合对比

上一节我们理解了融合算子的实现。本节运行两组 profiling 训练——**不启用融合算子**（baseline）和**启用融合算子**（fused），然后用 `analyze_profile.py` 自动对比。

---

## Profiling 配置说明

在 `config_registry.py` 中，profiling 相关配置集中在 `ProfilingConfig`：


下面所有 Bash 命令均在 torchtitan-npu 仓库根目录下执行。运行此 cell 完成工作目录切换和 CANN 环境配置：

In [ ]:
# 切换工作路径
import os
os.chdir(os.path.expanduser('~/torchtitan-npu'))
os.environ['BASH_ENV'] = os.path.expanduser('~/Ascend/cann/set_env.sh')
print(f'Working directory: {os.getcwd()}')


```python
profiling=ProfilingConfig(
    enable_profiling=False,      # ← 默认关闭，profiling 时改为 True
    enable_online_parse=True,    # 训练结束后自动解析 trace
    profile_ranks=[0],           # profile rank 0
    profile_step_start=5,        # 从第 5 步开始采集
    profile_step_end=6,          # 到第 6 步结束（采集 1 步）
    profile_record_shapes=True,  # 记录张量形状
    profile_with_memory=True,    # 记录显存使用
)
```


> **`profile_step_start=5`**：跳过前 5 步（含初始化、第一次 CUDA kernel 编译等），从第 6 步采集——此时 kernel 已预热，数据更能反映稳态性能。

### Profiling 的 batch_size 设置

第 3 章训练用 `global_batch_size=64` 追求收敛质量。Profiling 时可选择改为 `gbs=4`：


```text
gbs=64 → 32 次前向，梯度累计一次 → 追求收敛质量，慢但模型收敛好
gbs=4  →  2 次前向，梯度累计一次 → 追求跑得快，只看算子耗时
```


见以下训练配置中的  --training.global_batch_size=4 。

> Profiling 只关心算子的**单步耗时分布**，不关心收敛效果。用小 batch_size 减少等待时间，快速出 profile 结果。

---

## 训练命令

### baseline 配置

在 `config_registry.py` 中，使用不含 `model_converters` 的配置：


```python
def _qwen3_1_7b_converters() -> ModelConvertersContainer.Config:
    return ModelConvertersContainer.Config(
        converters=[
          # 基线 - 不启用任何 converters
        ],
    )
```


启动训练 + profiling：


In [ ]:
%%bash
grep -A3 'def _qwen3_1_7b_converters' torchtitan_npu/models/qwen3/config_registry.py

In [ ]:
%%bash
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh \
  --training.steps=10 \
  --training.global_batch_size=4 \
  --profiling.enable_profiling \
  --dataloader.dataset_path ./assets/data/wordle



### fused 配置

同一份 `config_registry.py`，启用 `model_converters`：


```python
def _qwen3_1_7b_converters() -> ModelConvertersContainer.Config:
    return ModelConvertersContainer.Config(
        converters=[
          # 启用 npu_rms_norm
          get_model_converter_config("npu_rms_norm"),
        ],
    )
```

修改配置：

In [ ]:
import re
path = 'torchtitan_npu/models/qwen3/config_registry.py'
with open(path) as f:
    c = f.read()
c = re.sub(
    r'(def _qwen3_1_7b_converters.*?converters=\[)[^\]]*(\])',
    r'\1\n          get_model_converter_config("npu_rms_norm"),\n        \2',
    c, flags=re.DOTALL
)
with open(path, 'w') as f:
    f.write(c)
print('Configured: fused (npu_rms_norm enabled)')

# 确认改动已被应用
!grep -A5 'def _qwen3_1_7b_converters' torchtitan_npu/models/qwen3/config_registry.py

启动训练：

In [ ]:
%%bash
NGPU=2 \s
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle\
bash scripts/run_train.sh \
  --training.steps=10 \
  --training.global_batch_size=4 \
  --profiling.enable_profiling \
  --dataloader.dataset_path ./assets/data/wordle




### 关键差异

两次运行唯一的不同是 `model_converters`：

| 配置 | model_converters | 效果 |
|------|-----------------|------|
| baseline | `[]`（空） | 原生 PyTorch RMSNorm，6 kernel/次 |
| fused | `["npu_rms_norm"]` | NPU 融合 RMSNorm，1 kernel/次 |

两次运行后，profiling 输出分别落在 `outputs/profile_traces/` 下的两个子目录中（目录名带时间戳）。

---

## 运行 analyze_profile.py

用以下命令自动发现并对比两组 trace：


In [ ]:
#!/usr/bin/env python3
"""自动发现并对比 baseline 与 fused 的 profiling trace。"""

import json, sys
from collections import defaultdict
from pathlib import Path

PROFILE_DIR = "./outputs/profile_traces"

# 关注的算子模式（子串匹配）
TARGET_PATTERNS = [
    ("aten::rms_norm",          "aten::rms_norm (PyTorch原生)"),
    ("aten::_fused_rms_norm",   "aten::_fused_rms_norm (PyTorch)"),
    ("npu_rms_norm",            "npu_rms_norm (NPU融合)"),
]
FRAGMENTED_ACL = [
    "AscendCL@aclnnPowTensorScalar",
    "AscendCL@aclnnMul",
    "AscendCL@aclnnInplaceMul",
]


def _safe_dur(evt: dict) -> float:
    d = evt.get("dur", 0)
    if isinstance(d, str):
        try: return float(d)
        except (ValueError, TypeError): return 0.0
    return float(d)


def discover_traces(profile_dir: Path) -> list[Path]:
    print(f"🔍 扫描: {profile_dir}")
    traces = []
    for subdir in sorted(profile_dir.iterdir()):
        if not subdir.is_dir():
            continue
        tf = subdir / "ASCEND_PROFILER_OUTPUT" / "trace_view.json"
        try:
            if tf.exists():
                traces.append((tf.stat().st_mtime, tf))
                print(f"   发现: {subdir.name}")
        except PermissionError:
            continue
    if len(traces) < 2:
        print(f"❌ 只找到 {len(traces)} 个 trace，需要至少 2 个")
        sys.exit(1)
    traces.sort(key=lambda t: t[0])
    return [t[1] for t in traces]


def load_trace(trace_path: Path) -> list[dict]:
    name = f"{trace_path.parent.parent.name}/{trace_path.parent.name}"
    with open(trace_path) as f:
        data = json.load(f)
    events = data if isinstance(data, list) else data.get("traceEvents", data.get("trace_events", []))
    print(f"   加载: {name}  ({len(events):,} events, {trace_path.stat().st_size/1024/1024:.0f} MiB)")
    return events


def aggregate_stats(events: list[dict]) -> dict:
    cpu_times, cpu_counts = defaultdict(float), defaultdict(int)
    acl_counts = defaultdict(int)
    total_events, total_time, acl_total = 0, 0.0, 0
    has_npu = False
    target_ops = {}

    for evt in events:
        name = evt.get("name", "")
        cat = evt.get("cat", "")
        dur = _safe_dur(evt)
        if dur <= 0 or not name:
            continue
        if cat == "cpu_op":
            cpu_times[name] += dur
            cpu_counts[name] += 1
            total_events += 1
            total_time += dur
        elif name.startswith("AscendCL@aclnn"):
            acl_counts[name] += 1
            acl_total += 1

    # 子串匹配：找到包含目标模式的算子并汇总
    for pattern, label in TARGET_PATTERNS:
        matched_time = 0.0
        matched_calls = 0
        for name in cpu_times:
            if pattern in name:
                matched_time += cpu_times[name]
                matched_calls += cpu_counts[name]
                if "npu_rms_norm" in pattern:
                    has_npu = True
        if matched_calls > 0:
            target_ops[label] = {"calls": matched_calls, "time_us": matched_time}

    frag_pow = sum(acl_counts.get(x, 0) for x in FRAGMENTED_ACL if "Pow" in x)
    frag_mul = sum(acl_counts.get(x, 0) for x in FRAGMENTED_ACL if "Mul" in x)

    return {
        "total_events": total_events, "total_time_us": total_time,
        "acl_launches": acl_total, "frag_pow": frag_pow, "frag_mul": frag_mul,
        "has_npu": has_npu, "target_ops": target_ops,
    }


def fmt_delta(before, after, is_int=True):
    d = after - before
    return f"{d:+,}" if is_int else f"{d:+,.1f}"


def fmt_pct(before, after):
    return "—" if before == 0 else f"{(after - before) / before * 100:+.1f}%"


def print_table(b: dict, f: dict, label_b: str, label_f: str):
    print()
    print(f"{"═" * 78}")
    print(f"  RMSNorm 融合算子 Profiling 对比")
    print(f"{"═" * 78}")
    print(f"  baseline: {label_b}")
    print(f"  fused:    {label_f}")
    print(f"  {"─" * 78}")

    hdr = f"  {"指标":<34s} {"BASELINE":>13s} {"FUSED":>13s} {"变化":>13s} {"变化%":>9s}"
    sep = f"  {"─" * 34}─{"─" * 13}─{"─" * 13}─{"─" * 13}─{"─" * 9}"
    print(hdr)
    print(sep)

    for label, bv, fv, is_int in [
        ("算子事件总数", b["total_events"], f["total_events"], True),
        ("ACL kernel 调用", b["acl_launches"], f["acl_launches"], True),
        ("总耗时 (ms)", b["total_time_us"]/1000, f["total_time_us"]/1000, False),
    ]:
        b_s = f"{bv:>13,}" if is_int else f"{bv:>13,.1f}"
        f_s = f"{fv:>13,}" if is_int else f"{fv:>13,.1f}"
        print(f"  {label:<34s} {b_s} {f_s} {fmt_delta(bv, fv, is_int):>13s} {fmt_pct(bv, fv):>9s}")

    # RMSNorm 算子
    print(sep)
    print(f"  {"── RMSNorm 算子耗时 ──":<34s}")
    for label in [l for _, l in TARGET_PATTERNS]:
        bo = b["target_ops"].get(label, {})
        fo = f["target_ops"].get(label, {})
        bt, ft = bo.get("time_us", 0), fo.get("time_us", 0)
        bc, fc = bo.get("calls", 0), fo.get("calls", 0)
        b_s = f"{bt:,.0f}μs ({bc}c)" if bc else "—"
        f_s = f"{ft:,.0f}μs ({fc}c)" if fc else "—"
        d_s = fmt_delta(bt, ft, is_int=False) + "μs" if (bt or ft) else "—"
        p_s = fmt_pct(bt, ft) if bt else "—"
        print(f"  {label:<34s} {b_s:>13s} {f_s:>13s} {d_s:>13s} {p_s:>9s}")

    # 碎片化
    print(sep)
    print(f"  {"── 碎片化 ACL kernel ──":<34s}")
    for label, bv, fv in [
        ("AscendCL@aclnnPow (Pow碎片)", b["frag_pow"], f["frag_pow"]),
        ("AscendCL@aclnnMul (Mul碎片)", b["frag_mul"], f["frag_mul"]),
    ]:
        print(f"  {label:<34s} {bv:>13,} {fv:>13,} {fmt_delta(bv, fv):>13s} {fmt_pct(bv, fv):>9s}")

    # 判定
    print(sep)
    bs = "✅" if b["has_npu"] else "❌"
    fs = "✅" if f["has_npu"] else "❌"
    print(f'  {"npu_rms_norm (NPU融合)":<34s} {bs:>13s} {fs:>13s} {"—":>13s} {"—":>9s}')
    print()
    if not b["has_npu"] and f["has_npu"]:
        print("  ✅ 融合正确生效: baseline 用 PyTorch RMSNorm → fused 用 NPU 融合算子")
    elif b["has_npu"]:
        print("  ⚠️ baseline 中已有 npu_rms_norm，可能两次都跑了 fused")
    else:
        print("  ❌ fused 中未检测到 npu_rms_norm，请检查 model_converters")
    print(f"  {"═" * 78}")


# ─── main ───
print()
print("🔍 Ascend NPU 融合算子 Profiling 分析工具")

profile_dir = Path(PROFILE_DIR)
traces = discover_traces(profile_dir)
bl_trace, fu_trace = traces[0], traces[-1]
bl_name = bl_trace.parent.parent.name
fu_name = fu_trace.parent.parent.name
print(f"   baseline: {bl_name}")
print(f"   fused:    {fu_name}")

bl_events = load_trace(bl_trace)
fu_events = load_trace(fu_trace)

bl_stats = aggregate_stats(bl_events)
fu_stats = aggregate_stats(fu_events)

print_table(bl_stats, fu_stats, bl_name, fu_name)
print("✅ 分析完成。")
print()



脚本会自动：

1. **发现 trace**：扫描 `profile_traces/` 下的所有子目录，找到 `ASCEND_PROFILER_OUTPUT/trace_view.json`。
2. **排序**：按文件修改时间排序——先跑的 = baseline，后跑的 = fused。
3. **解析**：加载 JSON，按算子名分类，统计 ACL kernel 调用次数和耗时。
4. **对比**：输出差异表格，验证融合算子是否生效。

### 预期输出


```text
═══════════════════════════════════════════════════════
  Ascend NPU 融合算子 Profiling 分析工具
═══════════════════════════════════════════════════════
🔍 扫描 profiling 输出目录: ./outputs/profile_traces
  📋 发现 trace: profile_baseline_xxx (时间戳: ...)
  📋 发现 trace: profile_fused_xxx (时间戳: ...)

  ✅ 自动排序结果（按时间）:
    1. baseline: profile_baseline_xxx
    2. fused:    profile_fused_xxx

✅ 分析完成。
```


下一节，我们将深入解读这些对比输出的含义。

## 练习

1. （判断题）Profiling 时 `profile_step_start=5` 是为了跳过前 5 步的 kernel 首次编译、显存预热等一次性开销，采集稳态性能数据。

2. （判断题）Profiling 时把 `global_batch_size` 从 64 改成 4，目的是减少单步训练时间以提高收敛质量。

3. （单选题）`analyze_profile.py` 如何自动区分 baseline 和 fused trace？
    A. 读取 trace 文件名中的 "baseline"/"fused" 关键字
    B. 按 `trace_view.json` 的文件修改时间排序——先跑的 = baseline，后跑的 = fused
    C. 比较两个 trace 的 loss 值
    D. 随机指定

In [ ]:
!cat ./answer/04.03_answer.txt